In [58]:
%pip install Faker pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [57]:

import os
import random
import numpy as np
import pandas as pd
from faker import Faker

# Folder chứa các file CSV
DATA_DIR = r"D:\L-NMPTDL&AI\Stage 1 + 2\DA-AI_Nhom8\DHHTTT20B_TCTA_Nhom8_Stage1\data"

# Tạo folder nếu chưa tồn tại
os.makedirs(DATA_DIR, exist_ok=True)

# Khởi tạo random seed để có thể tái lập kết quả
random.seed(42)
np.random.seed(42)
Faker.seed(42)

fake = Faker("en_US")

print("Data folder:", DATA_DIR)


Data folder: D:\L-NMPTDL&AI\Stage 1 + 2\DA-AI_Nhom8\DHHTTT20B_TCTA_Nhom8_Stage1\data


In [59]:

# Đọc dữ liệu từ hai nguồn
books_df = pd.read_csv(
    os.path.join(DATA_DIR, "books_crawled.csv")
)

shopee_df = pd.read_csv(
    os.path.join(DATA_DIR, "shopee_data.csv")
)

print("Books:", books_df.shape)
print("Shopee:", shopee_df.shape)

print("Books columns:", books_df.columns.tolist())
print("Shopee columns:", shopee_df.columns.tolist())


Books: (300, 5)
Shopee: (20312, 20)
Books columns: ['title', 'price', 'rating', 'availability', 'url']
Shopee columns: ['price_ori', 'delivery', 'item_category_detail', 'specification', 'title', 'w_date', 'link_ori', 'item_rating', 'seller_name', 'idElastic', 'price_actual', 'sitename', 'idHash', 'total_rating', 'id', 'total_sold', 'pict_link', 'favorite', 'timestamp', 'desc']


In [60]:

# Tạo ID cho từng sản phẩm trong mỗi nguồn
books_products = pd.DataFrame({
    "product_id": [
        f"BOOK{i:06d}" for i in range(1, len(books_df) + 1)
    ],
    "title": books_df["title"].astype("string"),
    "source": "Books to Scrape"
})

shopee_products = pd.DataFrame({
    "product_id": [
        f"SHOP{i:06d}" for i in range(1, len(shopee_df) + 1)
    ],
    "title": shopee_df["title"].astype("string"),
    "source": "Shopee"
})

# Chỉ tạo danh sách tham chiếu tạm thời, chưa concat hai bảng static
product_ids = (
    books_products["product_id"].tolist()
    + shopee_products["product_id"].tolist()
)

print("Total product IDs:", len(product_ids))
print("Sample IDs:", product_ids[:5])


Total product IDs: 20612
Sample IDs: ['BOOK000001', 'BOOK000002', 'BOOK000003', 'BOOK000004', 'BOOK000005']


In [61]:

N_INTERACTIONS = 1000

# Chọn ngẫu nhiên sản phẩm từ hai nguồn
selected_products = np.random.choice(
    product_ids,
    size=N_INTERACTIONS,
    replace=True
)

# Tạo thời gian tương tác theo từng giờ
timestamps = pd.date_range(
    start="2026-01-01",
    periods=N_INTERACTIONS,
    freq="h"
)

# Sinh dữ liệu động
dynamic_df = pd.DataFrame({
    "interaction_id": [
        f"INT{i:06d}" for i in range(1, N_INTERACTIONS + 1)
    ],
    "product_id": selected_products,
    "customer_id": [
        f"CUST{i:05d}"
        for i in np.random.randint(1, 501, size=N_INTERACTIONS)
    ],
    "customer_name": [
        fake.name() for _ in range(N_INTERACTIONS)
    ],
    "customer_email": [
        fake.email() for _ in range(N_INTERACTIONS)
    ],
    "customer_city": [
        fake.city() for _ in range(N_INTERACTIONS)
    ],
    "timestamp": timestamps,
    "views": np.random.randint(1, 21, size=N_INTERACTIONS),
    "add_to_cart": np.random.choice(
        [0, 1], size=N_INTERACTIONS, p=[0.7, 0.3]
    ),
    "purchase": np.random.choice(
        [0, 1], size=N_INTERACTIONS, p=[0.8, 0.2]
    ),
    "quantity": np.random.randint(1, 4, size=N_INTERACTIONS)
})

# Nếu không mua thì số lượng mua bằng 0
dynamic_df.loc[
    dynamic_df["purchase"] == 0,
    "quantity"
] = 0

print("Dynamic data shape:", dynamic_df.shape)
display(dynamic_df.head())


Dynamic data shape: (1000, 11)


,interaction_id,product_id,customer_id,customer_name,customer_email,customer_city,timestamp,views,add_to_cart,purchase,quantity
0,INT000001,SHOP015496,CUST00171,Allison Hill,qjacobson@example.org,West Alanchester,2026-01-01 00:00:00,20,0,0,0
1,INT000002,SHOP000561,CUST00460,Noah Rhodes,ylopez@example.com,Allisonshire,2026-01-01 01:00:00,15,1,1,1
2,INT000003,SHOP005091,CUST00492,Angie Henderson,seanbaker@example.com,Benjaminfurt,2026-01-01 02:00:00,7,0,0,0
3,INT000004,SHOP011665,CUST00356,Daniel Wagner,jeremy49@example.org,Palmerfort,2026-01-01 03:00:00,11,0,0,0
4,INT000005,SHOP010985,CUST00496,Cristian Santos,karencontreras@example.org,Lake Bruce,2026-01-01 04:00:00,20,0,0,0


In [62]:

# Kiểm tra kiểu dữ liệu
print(dynamic_df.dtypes)

# Kiểm tra dữ liệu thiếu
print("\nMissing values:")
print(dynamic_df.isna().sum())

# Thống kê lượt mua
print("\nPurchase counts:")
print(dynamic_df["purchase"].value_counts())

# Kiểm tra số lượng tương tác theo thành phố
display(
    dynamic_df["customer_city"]
    .value_counts()
    .head(10)
)


interaction_id            object
product_id                object
customer_id               object
customer_name             object
customer_email            object
customer_city             object
timestamp         datetime64[ns]
views                      int32
add_to_cart                int64
purchase                   int64
quantity                   int32
dtype: object

Missing values:
interaction_id    0
product_id        0
customer_id       0
customer_name     0
customer_email    0
customer_city     0
timestamp         0
views             0
add_to_cart       0
purchase          0
quantity          0
dtype: int64

Purchase counts:
purchase
0    816
1    184
Name: count, dtype: int64


customer_city
East Nicholas      3
West John          3
West James         3
West Jennifer      3
North Jasontown    2
Kellytown          2
Johnsontown        2
South Heather      2
North Mark         2
Jasonfort          2
Name: count, dtype: int64

In [63]:

DYNAMIC_PATH = os.path.join(
    DATA_DIR,
    "dynamic_data.csv"
)

dynamic_df.to_csv(
    DYNAMIC_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu file tại:")
print(DYNAMIC_PATH)


Đã lưu file tại:
D:\L-NMPTDL&AI\Stage 1 + 2\DA-AI_Nhom8\DHHTTT20B_TCTA_Nhom8_Stage1\data\dynamic_data.csv
